# Jersey City Citibike — Gold Layer Insights

Datenbasis: Jersey City Citibike, **August 2018 – Januar 2019** (6 Monate, ~187.000 Fahrten).
Februar/März 2019 sind bewusst zurückgehalten für spätere Airflow/CI-CD-Tests.

Quelle: `marts` / `marts_aggregates` Schema in Snowflake (Gold Layer).

## Setup — Verbindung zu Snowflake

In [1]:
import os
from dotenv import load_dotenv
from cryptography.hazmat.primitives import serialization
import snowflake.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap

load_dotenv()

with open(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"], "rb") as key_file:
    p_key = serialization.load_pem_private_key(
        key_file.read(),
        password=None,
    )

pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    private_key=pkb,
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="marts_aggregates"
)

print("Verbindung zu Snowflake erfolgreich.")

Verbindung zu Snowflake erfolgreich.


## Daten laden

Alle vier Aggregations-/Fact-Quellen aus dem Gold Layer auf einmal ziehen, damit nicht in jedem Abschnitt erneut zur Datenbank gewechselt werden muss.

In [2]:
# Stundenmuster nach Nutzertyp (Subscriber vs. Customer)
query_hourly = "SELECT * FROM jersey_city_bikeshare.marts_aggregates.agg_usage_patterns_by_hour"
df = pd.read_sql(query_hourly, conn)

# Tagesaggregation (Gesamtvolumen, Ø Dauer, etc.)
query_daily = "SELECT * FROM jersey_city_bikeshare.marts_aggregates.agg_daily_summary ORDER BY date_day"
df_daily = pd.read_sql(query_daily, conn)

# Stations-Ranking
query_stations = "SELECT * FROM jersey_city_bikeshare.marts_aggregates.agg_station_ranking ORDER BY total_activity DESC LIMIT 15"
df_stations = pd.read_sql(query_stations, conn)

# Einzelne Trips für Verteilungsanalysen (nur < 60 Min, um Ausreißer fürs Histogramm zu kappen)
query_trips = "SELECT trip_duration_seconds, user_type FROM jersey_city_bikeshare.marts.fct_trips WHERE trip_duration_seconds < 3600"
df_trips = pd.read_sql(query_trips, conn)

# Stationskoordinaten + Gesamtaktivität, für die Stations-Karte
query_station_coords = """
    SELECT
        s.station_id,
        s.station_name,
        s.station_latitude,
        s.station_longitude,
        COALESCE(r.total_activity, 0) AS total_activity
    FROM jersey_city_bikeshare.marts.dim_stations s
    LEFT JOIN jersey_city_bikeshare.marts_aggregates.agg_station_ranking r
        ON s.station_name = r.station_name
    WHERE s.station_latitude IS NOT NULL AND s.station_longitude IS NOT NULL
"""
df_station_coords = pd.read_sql(query_station_coords, conn)

# Stichprobe einzelner Fahrten inkl. Start-/End-Koordinaten, für die Fahrt-Linien-Karte
query_trip_routes = """
    SELECT
        f.trip_id,
        f.user_type,
        f.trip_duration_seconds,
        s1.station_name AS start_station_name,
        s1.station_latitude AS start_lat,
        s1.station_longitude AS start_lon,
        s2.station_name AS end_station_name,
        s2.station_latitude AS end_lat,
        s2.station_longitude AS end_lon
    FROM jersey_city_bikeshare.marts.fct_trips f
    JOIN jersey_city_bikeshare.marts.dim_stations s1 ON f.start_station_id = s1.station_id
    JOIN jersey_city_bikeshare.marts.dim_stations s2 ON f.end_station_id = s2.station_id
    WHERE f.start_station_id != f.end_station_id
    ORDER BY RANDOM()
    LIMIT 300
"""
df_trip_routes = pd.read_sql(query_trip_routes, conn)

print(f"Stundenmuster: {len(df):,} Zeilen")
print(f"Tagesdaten: {len(df_daily):,} Zeilen ({df_daily['DATE_DAY'].min()} bis {df_daily['DATE_DAY'].max()})")
print(f"Stationen: {len(df_stations):,} Zeilen")
print(f"Einzeltrips (<60min): {len(df_trips):,} Zeilen")
print(f"Stationskoordinaten: {len(df_station_coords):,} Zeilen")
print(f"Fahrt-Routen (Stichprobe): {len(df_trip_routes):,} Zeilen")
df.head()

/var/folders/2s/rp_j2lfd2358ncpst6b5csh00000gn/T/ipykernel_85593/1361769117.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_hourly, conn)
/var/folders/2s/rp_j2lfd2358ncpst6b5csh00000gn/T/ipykernel_85593/1361769117.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_daily = pd.read_sql(query_daily, conn)
/var/folders/2s/rp_j2lfd2358ncpst6b5csh00000gn/T/ipykernel_85593/1361769117.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_stations = pd.read_sql(query_stations, conn)
/var/folders/2s/r

Stundenmuster: 335 Zeilen
Tagesdaten: 243 Zeilen (2018-08-01 bis 2019-03-31)
Stationen: 15 Zeilen
Einzeltrips (<60min): 227,620 Zeilen
Stationskoordinaten: 90 Zeilen
Fahrt-Routen (Stichprobe): 300 Zeilen


,USER_TYPE,HOUR_OF_DAY,DAY_OF_WEEK,IS_WEEKEND,TRIP_COUNT,AVG_DURATION_SECONDS
0,Customer,0,0,True,37,3860.216216
1,Customer,1,0,True,25,1854.400000
2,Customer,2,0,True,29,5743.586207
3,Customer,3,0,True,16,2824.187500
4,Customer,4,0,True,11,906.454545


## Insight 1 — Fahrtenmuster nach Uhrzeit & Wochentag (absolute Zahlen)

Getrennte Farbskalen pro Nutzertyp, damit das kleinere Customer-Volumen nicht von Subscriber überdeckt wird.

In [3]:
df_sub = df[df["USER_TYPE"] == "Subscriber"]
df_cust = df[df["USER_TYPE"] == "Customer"]

pivot_sub = df_sub.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="TRIP_COUNT", aggfunc="sum", fill_value=0)
pivot_cust = df_cust.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="TRIP_COUNT", aggfunc="sum", fill_value=0)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Subscriber (Pendler)", "Customer (Freizeit)"),
    horizontal_spacing=0.15
)

fig.add_trace(
    go.Heatmap(
        z=pivot_sub.values, x=pivot_sub.columns, y=pivot_sub.index,
        colorscale="Viridis", colorbar=dict(title="Trips", x=0.46, len=0.9)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=pivot_cust.values, x=pivot_cust.columns, y=pivot_cust.index,
        colorscale="Plasma", colorbar=dict(title="Trips", x=1.02, len=0.9)
    ),
    row=1, col=2
)

fig.update_layout(
    title="Fahrtenmuster nach Uhrzeit & Wochentag (Aug 2018 – Jan 2019) — getrennte Skalen",
    height=450, width=1100
)
fig.update_xaxes(title_text="Uhrzeit", row=1, col=1)
fig.update_xaxes(title_text="Uhrzeit", row=1, col=2)
fig.update_yaxes(title_text="Wochentag (0=So, 6=Sa)", row=1, col=1)
fig.update_yaxes(title_text="Wochentag (0=So, 6=Sa)", row=1, col=2)
fig.show()

## Insight 2 — Relatives Nutzungsmuster (% des eigenen Gesamtvolumens)

Beide Heatmaps auf 100% normiert (innerhalb der jeweiligen Nutzergruppe), damit Form-Unterschiede statt nur Mengen-Unterschiede sichtbar werden.

In [4]:
df_sub_norm = df_sub.copy()
df_sub_norm["PCT_OF_TYPE"] = df_sub_norm["TRIP_COUNT"] / df_sub_norm["TRIP_COUNT"].sum() * 100

df_cust_norm = df_cust.copy()
df_cust_norm["PCT_OF_TYPE"] = df_cust_norm["TRIP_COUNT"] / df_cust_norm["TRIP_COUNT"].sum() * 100

pivot_sub_norm = df_sub_norm.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="PCT_OF_TYPE", aggfunc="sum", fill_value=0)
pivot_cust_norm = df_cust_norm.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="PCT_OF_TYPE", aggfunc="sum", fill_value=0)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Subscriber — Anteil am eigenen Volumen (%)", "Customer — Anteil am eigenen Volumen (%)"),
    horizontal_spacing=0.15
)

fig.add_trace(
    go.Heatmap(
        z=pivot_sub_norm.values, x=pivot_sub_norm.columns, y=pivot_sub_norm.index,
        colorscale="Viridis", colorbar=dict(title="% Anteil", x=0.46, len=0.9)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=pivot_cust_norm.values, x=pivot_cust_norm.columns, y=pivot_cust_norm.index,
        colorscale="Plasma", colorbar=dict(title="% Anteil", x=1.02, len=0.9)
    ),
    row=1, col=2
)

fig.update_layout(
    title="Relatives Nutzungsmuster (% des eigenen Gesamtvolumens) — vergleichbare Skalen",
    height=450, width=1150
)
fig.update_xaxes(title_text="Uhrzeit", row=1, col=1)
fig.update_xaxes(title_text="Uhrzeit", row=1, col=2)
fig.update_yaxes(title_text="Wochentag (0=So, 6=Sa)", row=1, col=1)
fig.update_yaxes(title_text="Wochentag (0=So, 6=Sa)", row=1, col=2)
fig.show()

## Insight 3 — Tägliches Fahrtenvolumen über die Zeit

Jetzt über den vollen 6-Monats-Zeitraum (Aug 2018 – Jan 2019), statt nur November. Saisonale Effekte (z.B. Kälteeinbruch im Winter) sollten hier sichtbar werden.

In [5]:
fig = px.line(
    df_daily,
    x="DATE_DAY",
    y="TOTAL_TRIPS",
    title="Tägliches Fahrtenvolumen (Aug 2018 – Jan 2019)",
    labels={"DATE_DAY": "Datum", "TOTAL_TRIPS": "Anzahl Fahrten"},
    markers=True
)
fig.update_layout(height=420, width=1050)
fig.show()

## Insight 4 — Durchschnittliche Fahrtdauer über die Zeit

In [6]:
fig = px.line(
    df_daily,
    x="DATE_DAY",
    y="AVG_TRIP_DURATION_SECONDS",
    title="Durchschnittliche Fahrtdauer pro Tag (Aug 2018 – Jan 2019)",
    labels={"DATE_DAY": "Datum", "AVG_TRIP_DURATION_SECONDS": "Ø Dauer (Sekunden)"},
    markers=True
)
fig.update_layout(height=420, width=1050)
fig.show()

## Insight 5 — Top 15 Stationen nach Gesamtaktivität

Start + Ende zusammen, über den vollen 6-Monats-Zeitraum.

In [7]:
fig = px.bar(
    df_stations.sort_values("TOTAL_ACTIVITY"),
    x="TOTAL_ACTIVITY",
    y="STATION_NAME",
    orientation="h",
    title="Top 15 Stationen nach Gesamtaktivität (Start + Ende), Aug 2018 – Jan 2019",
    labels={"TOTAL_ACTIVITY": "Anzahl Fahrten (Start+Ende)", "STATION_NAME": "Station"}
)
fig.update_layout(height=600, width=1000)
fig.show()

## Insight 5b — Stationskarte (Standorte + Aktivität)

Jede Station als Kreis: Größe und Farbe zeigen die Gesamtaktivität (Start + Ende), über den vollen 6-Monats-Zeitraum.

In [8]:
center_lat = df_station_coords["STATION_LATITUDE"].mean()
center_lon = df_station_coords["STATION_LONGITUDE"].mean()

m_stations = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

max_activity = df_station_coords["TOTAL_ACTIVITY"].max()

for _, row in df_station_coords.iterrows():
    activity_ratio = row["TOTAL_ACTIVITY"] / max_activity if max_activity > 0 else 0
    radius = 4 + activity_ratio * 26  # 4px für inaktive Stationen, bis zu 30px für die aktivste

    folium.CircleMarker(
        location=[row["STATION_LATITUDE"], row["STATION_LONGITUDE"]],
        radius=radius,
        popup=f"{row['STATION_NAME']}<br>{row['TOTAL_ACTIVITY']:,} Fahrten",
        tooltip=row["STATION_NAME"],
        color="#d73027" if activity_ratio > 0.5 else "#4575b4",
        fill=True,
        fill_opacity=0.6,
        weight=1
    ).add_to(m_stations)

m_stations

## Insight 5c — Fahrt-Routen (Stichprobe einzelner Fahrräder)

300 zufällige Fahrten als Linien von Start- zu Endstation (nur Fahrten, die tatsächlich die Station gewechselt haben). Farbe nach Nutzertyp — zeigt, ob Subscriber und Customer unterschiedliche Strecken-Korridore nutzen.

In [9]:
m_routes = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

color_map = {"Subscriber": "#377eb8", "Customer": "#e41a1c"}

for _, row in df_trip_routes.iterrows():
    folium.PolyLine(
        locations=[[row["START_LAT"], row["START_LON"]], [row["END_LAT"], row["END_LON"]]],
        color=color_map.get(row["USER_TYPE"], "#999999"),
        weight=1.5,
        opacity=0.4,
        tooltip=f"{row['START_STATION_NAME']} -> {row['END_STATION_NAME']} ({row['USER_TYPE']})"
    ).add_to(m_routes)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; background: white;
            padding: 10px; border-radius: 6px; border: 1px solid #999; font-size: 13px;">
  <b>Nutzertyp</b><br>
  <span style="color:#377eb8;">&#9644;&#9644;</span> Subscriber<br>
  <span style="color:#e41a1c;">&#9644;&#9644;</span> Customer
</div>
"""
m_routes.get_root().html.add_child(folium.Element(legend_html))

m_routes

## Insight 5d — Aktivitäts-Heatmap (alle Start-Standorte)

Statt einzelner Linien: alle Fahrt-Startpunkte als Dichte-Heatmap — zeigt, wo räumlich die meiste Aktivität ist, unabhängig von festen Stationsgrenzen.

In [10]:
m_heat = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

heat_data = df_station_coords[["STATION_LATITUDE", "STATION_LONGITUDE", "TOTAL_ACTIVITY"]].values.tolist()

HeatMap(heat_data, radius=25, blur=20, max_zoom=15).add_to(m_heat)

m_heat

## Insight 5e — Cross-System-Trips (Jersey City ↔ Manhattan/Brooklyn)

`dim_stations` enthält auch Stationen außerhalb von Jersey City, weil sie als Endpunkt von Fahrten auftauchen. Das deutet auf ein verbundenes Citibike-System hin (JC-Bike kann drüben abgestellt werden). Hier quantifizieren wir, wie groß dieser Anteil tatsächlich ist.

Grenze: Stationen mit Längengrad > -74.02 gelten als "jenseits des Hudson" (Manhattan/Brooklyn-Seite).

In [11]:
query_cross_system = """
    SELECT
        f.user_type,
        CASE
            WHEN s_end.station_longitude > -74.02 THEN 'Cross-System (endet drüben)'
            ELSE 'Jersey City intern'
        END AS trip_type,
        COUNT(*) AS trip_count
    FROM jersey_city_bikeshare.marts.fct_trips f
    JOIN jersey_city_bikeshare.marts.dim_stations s_end ON f.end_station_id = s_end.station_id
    GROUP BY f.user_type, trip_type
"""
df_cross_system = pd.read_sql(query_cross_system, conn)
df_cross_system

/var/folders/2s/rp_j2lfd2358ncpst6b5csh00000gn/T/ipykernel_85593/1568426730.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cross_system = pd.read_sql(query_cross_system, conn)


,USER_TYPE,TRIP_TYPE,TRIP_COUNT
0,Customer,Jersey City intern,12070
1,Subscriber,Jersey City intern,217396
2,Subscriber,Cross-System (endet drüben),59
3,Customer,Cross-System (endet drüben),29


In [12]:
totals_overall = df_cross_system["TRIP_COUNT"].sum()
df_overall = df_cross_system.groupby("TRIP_TYPE")["TRIP_COUNT"].sum().reset_index()

fig = px.pie(
    df_overall,
    names="TRIP_TYPE",
    values="TRIP_COUNT",
    title=f"Anteil Cross-System-Trips an allen Fahrten (gesamt: {totals_overall:,})",
    color="TRIP_TYPE",
    color_discrete_map={"Jersey City intern": "#4575b4", "Cross-System (endet drüben)": "#d73027"},
    hole=0.4
)
fig.update_traces(textinfo="percent+label")
fig.update_layout(height=420, width=600)
fig.show()

In [13]:
fig = px.bar(
    df_cross_system,
    x="USER_TYPE",
    y="TRIP_COUNT",
    color="TRIP_TYPE",
    barmode="group",
    title="Cross-System-Trips vs. JC-interne Trips, nach Nutzertyp",
    labels={"USER_TYPE": "Nutzertyp", "TRIP_COUNT": "Anzahl Fahrten", "TRIP_TYPE": "Trip-Typ"},
    color_discrete_map={"Jersey City intern": "#4575b4", "Cross-System (endet drüben)": "#d73027"},
    log_y=True
)
fig.update_layout(height=420, width=800)
fig.show()

## Insight 6 — Verteilung der Fahrtdauer (absolut & normalisiert)

Absolute Zähler zuerst (zeigt das tatsächliche Volumen), dann normalisiert auf Prozent (zeigt die Form der Verteilung unabhängig von der Gruppengröße).

In [14]:
fig = px.histogram(
    df_trips,
    x="TRIP_DURATION_SECONDS",
    color="USER_TYPE",
    nbins=60,
    title="Verteilung der Fahrtdauer (unter 60 Min) — absolute Zahlen",
    labels={"TRIP_DURATION_SECONDS": "Fahrtdauer (Sekunden)"},
    opacity=0.7,
    barmode="overlay"
)
fig.update_layout(height=420, width=1000)
fig.show()

In [15]:
fig = px.histogram(
    df_trips,
    x="TRIP_DURATION_SECONDS",
    color="USER_TYPE",
    nbins=60,
    title="Verteilung der Fahrtdauer — normalisiert (% je Nutzertyp)",
    labels={"TRIP_DURATION_SECONDS": "Fahrtdauer (Sekunden)"},
    opacity=0.6,
    barmode="overlay",
    histnorm="percent"
)
fig.update_layout(height=420, width=1000)
fig.show()

## Insight 7 — Fahrten pro Wochentag (absolut & normalisiert)

In [16]:
weekday_names = {0: "So", 1: "Mo", 2: "Di", 3: "Mi", 4: "Do", 5: "Fr", 6: "Sa"}
weekday_order = ["So", "Mo", "Di", "Mi", "Do", "Fr", "Sa"]

df_weekday = df.groupby(["DAY_OF_WEEK", "USER_TYPE"])["TRIP_COUNT"].sum().reset_index()
df_weekday["WEEKDAY_NAME"] = df_weekday["DAY_OF_WEEK"].map(weekday_names)

fig = px.bar(
    df_weekday,
    x="WEEKDAY_NAME",
    y="TRIP_COUNT",
    color="USER_TYPE",
    barmode="group",
    title="Fahrten pro Wochentag, nach Nutzertyp — absolute Zahlen",
    labels={"WEEKDAY_NAME": "Wochentag", "TRIP_COUNT": "Anzahl Fahrten"},
    category_orders={"WEEKDAY_NAME": weekday_order}
)
fig.update_layout(height=420, width=1000)
fig.show()

In [17]:
totals = df_weekday.groupby("USER_TYPE")["TRIP_COUNT"].sum().reset_index().rename(columns={"TRIP_COUNT": "TOTAL"})
df_weekday_pct = df_weekday.merge(totals, on="USER_TYPE")
df_weekday_pct["PCT"] = df_weekday_pct["TRIP_COUNT"] / df_weekday_pct["TOTAL"] * 100

fig = px.bar(
    df_weekday_pct,
    x="WEEKDAY_NAME",
    y="PCT",
    color="USER_TYPE",
    barmode="group",
    title="Anteil der Fahrten pro Wochentag (% des jeweiligen Gesamtvolumens)",
    labels={"WEEKDAY_NAME": "Wochentag", "PCT": "Anteil (%)"},
    category_orders={"WEEKDAY_NAME": weekday_order}
)
fig.update_layout(height=420, width=1000)
fig.show()

## Zusammenfassung

| Dimension | Subscriber (Pendler) | Customer (Freizeit) |
|---|---|---|
| Wann | Mo–Fr, 7–9 Uhr & 17–19 Uhr | Sa/So, 10–15 Uhr |
| Fahrtdauer | Kurz, eng verteilt | Länger, breit gestreut |
| Aktivste Station | Grove St PATH | — |
| Räumliches Muster | Siehe Stations- & Routenkarte (Insight 5b–5d) | Siehe Stations- & Routenkarte |

*Hinweis: Werte basieren auf 6 Monaten (Aug 2018 – Jan 2019). Feb/März 2019 sind zurückgehalten für spätere Pipeline-Tests.*

## Gesamtfazit — Datenanalyse Jersey City Citibike (Aug 2018 – Jan 2019)

### Datenbasis
- **240.538 Fahrten** über 6 Monate (August 2018 – Januar 2019), Jersey City Citibike-System
- Verarbeitet über eine vollständige Medallion-Architektur (Raw → Staging → Marts) in Snowflake, transformiert mit dbt
- **87 Stationen**, davon der überwiegende Teil im Kerngebiet von Jersey City
- Februar/März 2019 (42.171 zusätzliche Fahrten) bewusst zurückgehalten und später als simulierter "neue Daten"-Produktionslauf über eine Airflow/Cosmos-Pipeline nachgeladen — inkrementelles Laden funktionierte fehlerfrei (nur die neuen Zeilen wurden verarbeitet, nicht der komplette Datensatz neu berechnet)

### Kernbefund: Zwei klar unterscheidbare Nutzergruppen

**Subscriber (Abonnenten) — ca. 95% aller Fahrten**
- Klassisches **Pendlerverhalten**: zwei scharfe Aktivitätsspitzen morgens (7–9 Uhr) und abends (17–19 Uhr)
- Konzentriert auf **Werktage** (Mo–Fr deutlich höher als Sa/So)
- Kurze, eng verteilte Fahrtdauer (Peak bei ca. 250–300 Sekunden / 4–5 Minuten) — typische letzte-Meile-Strecken, z. B. zur PATH-Bahnstation
- Bleiben fast ausschließlich innerhalb des Jersey-City-Systems (nur 0,026 % der Fahrten enden außerhalb)

**Customer (Einzelfahrer) — ca. 5% aller Fahrten**
- **Freizeit-/Tourismusmuster**: stärkste Aktivität am Sonntag, 10–15 Uhr, mit zweitem kleineren Hotspot Samstagabend
- Fahrtdauer deutlich länger und breiter gestreut (signifikanter Anteil über 800–2000+ Sekunden) — explorative, weniger zielgerichtete Fahrten
- Anteilig ca. 9× häufiger "Cross-System" (enden außerhalb von Jersey City) als Subscriber — passt zum explorativeren Nutzungsprofil

### Räumliche Erkenntnisse
- **Grove St PATH** ist die mit Abstand aktivste Station (~7.000 Fahrten Start+Ende), gefolgt von Exchange Place, Hamilton Park und Sip Ave — alle in unmittelbarer Nähe zu Bahnanschlüssen, was die Pendler-Hypothese zusätzlich stützt
- Die Stationsdatenbank enthält auch einzelne Stationen in Manhattan/Brooklyn, da das Citibike-System dort technisch verbunden ist; tatsächlich genutzt wird diese Verbindung aber nur in **0,037 % aller Fahrten** — geografisch und operativ ist das System de facto auf Jersey City beschränkt

### Datenqualität
- Alle 14 automatisierten dbt-Tests (Eindeutigkeit, Null-Prüfungen, referenzielle Integrität zwischen Fact- und Dimensionstabellen) liefen über den gesamten Datensatz erfolgreich
- Im Rahmen der Entwicklung wurde ein realer Synchronisationsfehler zwischen `fct_trips` und `dim_dates` durch die automatisierten Tests aufgedeckt und behoben (View-Modell war nicht automatisch mit neuen Inkrement-Daten synchron)
- Pipeline wurde unter realistischer Lastveränderung getestet: Laufzeit blieb bei einer Steigerung von 24.910 auf 229.554 Zeilen (~9-fache Menge) nahezu konstant (~8 Sekunden Gesamtlaufzeit)

### Geschäftliche Einordnung
| Dimension | Subscriber (Pendler) | Customer (Freizeit) |
|---|---|---|
| Wann | Mo–Fr, 7–9 Uhr & 17–19 Uhr | Sa/So, 10–15 Uhr |
| Fahrtdauer | Kurz, eng verteilt (~4–5 Min) | Länger, breit gestreut |
| Wochenend-Anteil | ~18 % des Volumens | ~41 % des Volumens |
| Cross-System-Rate | 0,026 % | 0,23 % |

**Implikation für Pricing/Marketing:** Subscriber-Angebote sollten auf verlässliche, kurze Arbeitswege ausgerichtet sein (z. B. Jahres-/Monatsabos, Fokus auf Stationskapazität an Pendlerknoten zu Stoßzeiten). Customer-Angebote sollten auf Wochenend-/Freizeitnutzung zugeschnitten sein (Tagestickets, Tourismus-Marketing, Kapazitätsplanung an Freizeitstandorten am Wochenende).

---
*Analyse basiert auf öffentlichen Citibike-Trip-Daten (Jersey City), verarbeitet über eine selbst entwickelte dbt/Snowflake-Pipeline mit Airflow-Orchestrierung (Cosmos).*